In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
data_1_1 = pd.read_csv("homework_1.1.csv")

Linear regression on 1.1csv

In [3]:
X = data_1_1[['X1', 'X2', 'X3']]
y = data_1_1['Y']

model = LinearRegression()
model.fit(X, y)

print(f"Coefficients (X1, X2, X3): {model.coef_}")
print(f"Intercept: {model.intercept_}")

Coefficients (X1, X2, X3): [1.00713766 1.96456859 2.97548854]
Intercept: 0.0026430033444732604


In [4]:
# Simple regression coefficients
print("Simple Regression Coefficients:")
coefs_simple = {}
for col in ['X1', 'X2', 'X3']:
    model_simple = LinearRegression()
    model_simple.fit(data_1_1[[col]], data_1_1['Y'])
    coefs_simple[col] = model_simple.coef_[0]
    print(f"{col}: {coefs_simple[col]}")

print("\nMultiple Regression Coefficients:")
coefs_multiple = {'X1': model.coef_[0], 'X2': model.coef_[1], 'X3': model.coef_[2]}
for col in ['X1', 'X2', 'X3']:
    print(f"{col}: {coefs_multiple[col]}")
    
print("\nDifferences (Absolute):")
diffs = {}
for col in ['X1', 'X2', 'X3']:
    diffs[col] = abs(coefs_multiple[col] - coefs_simple[col])
    print(f"{col}: {diffs[col]}")

max_diff_col = max(diffs, key=diffs.get)
print(f"\nGreatest difference is for {max_diff_col}: {diffs[max_diff_col]}")

Simple Regression Coefficients:
X1: 1.8417610991715148
X2: 4.083612579423011
X3: 3.0970412020437936

Multiple Regression Coefficients:
X1: 1.0071376550759568
X2: 1.9645685948713516
X3: 2.9754885351434233

Differences (Absolute):
X1: 0.834623444095558
X2: 2.1190439845516598
X3: 0.12155266690037037

Greatest difference is for X2: 2.1190439845516598


In [5]:
import statsmodels.api as sm

# Add a constant for the intercept
X_with_const = sm.add_constant(X)
model_sm = sm.OLS(y, X_with_const).fit()

# Print the t-statistics
print(model_sm.tvalues)

const      0.166181
X1        60.984011
X2        53.283212
X3       196.645240
dtype: float64


In [6]:
data_1_2 = pd.read_csv("homework_1.2.csv")

In [ ]:
from sklearn.neighbors import NearestNeighbors

# Extract the Z variable for matching
Z_values = data_1_2[['Z']]

# NearestNeighbors
nn = NearestNeighbors(n_neighbors=2) 
nn.fit(Z_values)

# Find distances and indices of the nearestneighbors
distances, indices = nn.kneighbors(Z_values)

# Display some of the matches (ignoring the point matching itself)
print("Distances to nearest neighbor:\n", distances[:, 1][:10])
print("Indices of nearest neighbor:\n", indices[:, 1][:10])

Distances to nearest neighbor:
 [0.00393032 0.00113784 0.00208214 0.00393032 0.00899286 0.00597309
 0.00055526 0.02176085 0.0127967  0.01471635]
Indices of nearest neighbor:
 [ 3 93 73  0 29 25 41 19 70 65]


In [ ]:
from sklearn.neighbors import NearestNeighbors

# Separate data based on X
data_x0 = data_1_2[data_1_2['X'] == 0]
data_x1 = data_1_2[data_1_2['X'] == 1]

# NearestNeighbors for radius matching
nn_radius = NearestNeighbors(radius=0.2)
nn_radius.fit(data_x0[['Z']])

# Find all matches in X=0 for each X=1 within distance 0.2
distances_radius, indices_radius = nn_radius.radius_neighbors(data_x1[['Z']])

# Collect the matched X=0 samples
matched_x0_list = []
for i, x1_idx in enumerate(data_x1.index):
    # indices_radius[i] contains the positional indices in data_x0
    match_indices_in_x0 = indices_radius[i]
    if len(match_indices_in_x0) > 0:
        # Get the actual DataFrame indices from data_x0
        actual_indices = data_x0.iloc[match_indices_in_x0].index
        
        for idx in actual_indices:
            matched_x0_list.append(data_x0.loc[idx])

import pandas as pd
# DataFrame containing all matched X=0 samples (duplicates included)
matched_x0_df = pd.DataFrame(matched_x0_list)

print(f"Total X=1 samples: {len(data_x1)}")
print(f"Total matched X=0 samples (including duplicates): {len(matched_x0_df)}")
print(f"Number of unique matched X=0 samples: {matched_x0_df.index.nunique()}")


Total X=1 samples: 48
Total matched X=0 samples (including duplicates): 737
Number of unique matched X=0 samples: 52


In [ ]:
# Number of duplicate matches
total_matches = len(matched_x0_df)
unique_matches = matched_x0_df.index.nunique()
duplicates = total_matches - unique_matches

print(f"Total matched X=0 samples: {total_matches}")
print(f"Unique matched X=0 samples: {unique_matches}")
print(f"Number of duplicates (all but the first): {duplicates}")

Total matched X=0 samples: 737
Unique matched X=0 samples: 52
Number of duplicates (all but the first): 685


In [14]:
import numpy as np

effects = []

# Loop over each X=1 item and its neighbors
for i, x1_idx in enumerate(data_x1.index):
    match_indices_in_x0 = indices_radius[i]
    
    if len(match_indices_in_x0) > 0:
        # Get the DataFrame indices for the matched X=0 item(s)
        actual_indices = data_x0.iloc[match_indices_in_x0].index
        
        # Compute the mean Y for this neighbor group
        mean_y0 = data_x0.loc[actual_indices, 'Y'].mean()
        
        # Get the Y value for the current X=1 item
        y1 = data_x1.loc[x1_idx, 'Y']
        
        # Add the difference observed (Y1 - Y0) to our effects list
        effects.append(y1 - mean_y0)

# Calculate the overall average effect
average_effect = np.mean(effects)

print(f"Number of X=1 samples with at least one match: {len(effects)}")
print(f"The average treatment effect is: {average_effect}")

Number of X=1 samples with at least one match: 46
The average treatment effect is: 0.5688516534127853
